## Imperial Valley Subsidence, CA USA (2015)

The PyGMTSAR InSAR library, Geomed3D Geophysical Inversion Library, N-Cube 3D/4D GIS Data Visualization, among others, are my open-source projects developed in my free time. I hold a Master's degree in STEM, specializing in radio physics. In 2004, I received the first prize in the All-Russian Physics Competition for significant results in forward and inverse modeling for nonlinear optics and holography. These skills are also applicable to modeling Gravity, Magnetic, and Thermal fields, as well as satellite interferometry processing. With 20 years of experience as a data scientist and software developer, I have contributed to scientific and industrial development, working on government contracts, university projects, and with companies like LG Corp and Google Inc.

You can support my work on [Patreon](https://www.patreon.com/pechnikov), where I share updates on my projects, publications, use cases, examples, and other useful information. For research and development services and support, please visit my profile on the freelance platform [Upwork](https://www.upwork.com).

### Resources
- Google Colab Pro notebooks and articles on [Patreon](https://www.patreon.com/pechnikov),
- Google Colab notebooks on [GitHub](https://github.com),
- Docker Images on [DockerHub](https://hub.docker.com),
- Geological Models on [YouTube](https://www.youtube.com),
- VR/AR Geological Models on [GitHub](https://github.com),
- Live updates and announcements on [LinkedIn](https://www.linkedin.com/in/alexey-pechnikov/).

© Alexey Pechnikov, 2024

$\large\color{blue}{\text{Hint: Use menu Cell} \to \text{Run All or Runtime} \to \text{Complete All or Runtime} \to \text{Run All}}$
$\large\color{blue}{\text{(depending of your localization settings) to execute the entire notebook}}$

## Google Colab Installation

Install PyGMTSAR and required GMTSAR binaries (including SNAPHU)

In [ ]:
import platform, sys, os
if 'google.colab' in sys.modules:
    # install PyGMTSAR stable version from PyPI
    !{sys.executable} -m pip install -q pygmtsar
    # alternatively, nstall PyGMTSAR development version from GitHub
    #!{sys.executable} -m pip install -Uq git+https://github.com/mobigroup/gmtsar.git@pygmtsar2#subdirectory=pygmtsar
    # use PyGMTSAR Google Colab installation script to install binary dependencies
    # script URL: https://github.com/AlexeyPechnikov/pygmtsar/blob/pygmtsar2/pygmtsar/pygmtsar/data/google_colab.sh
    import importlib.resources as resources
    with resources.as_file(resources.files('pygmtsar.data') / 'google_colab.sh') as google_colab_script_filename:
        !sh {google_colab_script_filename}
    # enable custom widget manager as required by recent Google Colab updates
    from google.colab import output
    output.enable_custom_widget_manager()
    # initialize virtual framebuffer for interactive 3D visualization; required for headless environments
    import xvfbwrapper
    display = xvfbwrapper.Xvfb(width=800, height=600)
    display.start()

# specify GMTSAR installation path
PATH = os.environ['PATH']
if PATH.find('GMTSAR') == -1:
    PATH = os.environ['PATH'] + ':/usr/local/GMTSAR/bin/'
    %env PATH {PATH}

# display PyGMTSAR version
from pygmtsar import __version__
__version__

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.5 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)




update-alternatives: using /usr/bin/gcc-9 to provide /usr/bin/gcc (gcc) in auto mode
There is only one alternative in link group gcc (providing /usr/bin/gcc): /usr/bin/gcc-9
Noth

'2025.4.8.post1'

## Load and Setup Python Modules

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from dask.distributed import Client
import dask
import json
import warnings
import logging
# supress warning "Detected different `run_spec` for key"
logging.getLogger("distributed.scheduler").setLevel(logging.ERROR)

In [ ]:
# plotting modules
import pyvista as pv
# magic trick for white background
pv.set_plot_theme("document")
import panel
panel.extension(comms='ipywidgets')
panel.extension('vtk')
from contextlib import contextmanager
# plotting modules
import matplotlib.pyplot as plt
import matplotlib
@contextmanager
def mpl_settings(settings):
    original_settings = {k: plt.rcParams[k] for k in settings}
    plt.rcParams.update(settings)
    yield
    plt.rcParams.update(original_settings)
plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.titlesize'] = 24
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
from ipyleaflet import Map, GeoJSON, TileLayer, LayersControl, basemaps, Popup
from ipywidgets import HTML
from IPython.display import display
%matplotlib inline

In [ ]:
from pygmtsar import S1, Stack, tqdm_dask, ASF, Tiles

# recent Google Colab changes in early September 2025 broke Dask+Xarray NetCDF multithedded processing (again)
# workaround below disables multitheading when it does not work, degrading performance and increasing RAM usage.
if 'google.colab' in sys.modules:
    methods = {
        "load_dem":  "synchronous",
        "save_cube": "compute",
        "save_stack":"compute",
    }
    for m, kind in methods.items():
        if not hasattr(Stack, f"_{m}"):
            setattr(Stack, f"_{m}", getattr(Stack, m))
        def _make_wrapper(name, kind):
            orig = getattr(Stack, f"_{name}")
            if kind == "synchronous":
                def _wrapper(self, *args, **kwargs):
                    with dask.config.set(scheduler="synchronous"):
                        return orig(self, *args, **kwargs)
                return _wrapper
            elif kind == "compute":
                def _wrapper(self, *args, **kwargs):
                    if args:
                        return orig(self, args[0].compute() if hasattr(args[0], "compute") else args[0], *args[1:], **kwargs)
                    return orig(self, **kwargs)
                return _wrapper
            else:
                raise NotImplementedError(f"Unknown wrapper kind: {kind}")
        setattr(Stack, m, _make_wrapper(m, kind))

## Define Sentinel-1 SLC Scenes and Processing Parameters

In [ ]:
SCENES = ['S1A_IW_SLC__1SSV_20150121T134412_20150121T134426_004270_005317_DBBE',
          'S1A_IW_SLC__1SSV_20150310T134411_20150310T134426_004970_006386_36B8',
          'S1A_IW_SLC__1SSV_20150403T134412_20150403T134426_005320_006BC4_3A7A',
          'S1A_IW_SLC__1SSV_20150427T134413_20150427T134428_005670_00745C_C1D8',
          'S1A_IW_SLC__1SSV_20150521T134415_20150521T134429_006020_007C3F_13DD']
SUBSWATH = 1
POLARIZATION = 'VV'

In [ ]:
REFERENCE    = '2015-04-03'
WORKDIR      = 'raw_imperial'
DATADIR      = 'data_imperial'
BASEDAYS     = 100
BASEMETERS   = 150

In [ ]:
# define DEM filename inside data directory
DEM = f'{DATADIR}/dem.nc'

## Download and Unpack Datasets

## Enter Your ASF User and Password

If the data directory is empty or doesn't exist, you'll need to download Sentinel-1 scenes from the Alaska Satellite Facility (ASF) datastore. Use your Earthdata Login credentials. If you don't have an Earthdata Login, you can create one at https://urs.earthdata.nasa.gov//users/new

You can also use pre-existing SLC scenes stored on your Google Drive, or you can copy them using a direct public link from iCloud Drive.

The credentials below are available at the time the notebook is validated.

In [ ]:
# Set these variables to None and you will be prompted to enter your username and password below.
asf_username = 'GoogleColab2023'
asf_password = 'GoogleColab_2023'

In [ ]:
# Set these variables to None and you will be prompted to enter your username and password below.
asf = ASF(asf_username, asf_password)
# Optimized scene downloading from ASF - only the required subswaths and polarizations.
print(asf.download(DATADIR, SCENES, SUBSWATH))

ASF Downloading Sentinel-1 SLC Scenes::   0%|          | 0/5 [00:00<?, ?it/s]

                                      burst_or_scene
0  S1A_IW_SLC__1SSV_20150121T134412_20150121T1344...
1  S1A_IW_SLC__1SSV_20150310T134411_20150310T1344...
2  S1A_IW_SLC__1SSV_20150403T134412_20150403T1344...
3  S1A_IW_SLC__1SSV_20150427T134413_20150427T1344...
4  S1A_IW_SLC__1SSV_20150521T134415_20150521T1344...


In [ ]:
# scan the data directory for SLC scenes and download missed orbits
S1.download_orbits(DATADIR, S1.scan_slc(DATADIR))

,orbit
0,S1A_OPER_AUX_POEORB_OPOD_20210305T143838_V2015...
1,S1A_OPER_AUX_POEORB_OPOD_20210306T054130_V2015...
2,S1A_OPER_AUX_POEORB_OPOD_20210306T133136_V2015...
3,S1A_OPER_AUX_POEORB_OPOD_20210306T211838_V2015...
4,S1A_OPER_AUX_POEORB_OPOD_20210307T050740_V2015...


In [ ]:
# define AOI as the whole scenes area
AOI = S1.scan_slc(DATADIR)
# download Copernicus Global DEM 1 arc-second
Tiles().download_dem(AOI, filename=DEM).plot.imshow(cmap='cividis')

## Run Local Dask Cluster

Launch Dask cluster for local and distributed multicore computing. That's possible to process terabyte scale Sentinel-1 SLC datasets on Apple Air 16 GB RAM.

In [ ]:
# simple Dask initialization
if 'client' in globals():
    client.close()
client = Client()
client

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:46467'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:37207'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:42161'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:42923'
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:52498
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:52512
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:52520
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:52516
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:52528


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 12,Total memory: 52.96 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37045,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:40547,Total threads: 3
Dashboard: http://127.0.0.1:41611/status,Memory: 13.24 GiB
Nanny: tcp://127.0.0.1:46467,


## Init

Search recursively for measurement (.tiff) and annotation (.xml) and orbit (.EOF) files in the DATA directory. It can be directory with full unzipped scenes (.SAFE) subdirectories or just a directory with the list of pairs of required .tiff and .xml files (maybe pre-filtered for orbit, polarization and subswath to save disk space). If orbit files and DEM are missed these will be downloaded automatically below.

In [ ]:
scenes = S1.scan_slc(DATADIR)
scenes

,datetime,orbit,mission,polarization,subswath,datapath,metapath,noisepath,calibpath,orbitpath,geometry
date,,,,,,,,,,,
2015-01-21,2015-01-21 13:44:13,D,S1A,VV,1,data_imperial/S1A_IW_SLC__1SSV_20150121T134412...,data_imperial/S1A_IW_SLC__1SSV_20150121T134412...,None,None,data_imperial/S1A_OPER_AUX_POEORB_OPOD_2021030...,"MULTIPOLYGON (((-114.64126 32.6171, -114.69249..."
2015-03-10,2015-03-10 13:44:12,D,S1A,VV,1,data_imperial/S1A_IW_SLC__1SSV_20150310T134411...,data_imperial/S1A_IW_SLC__1SSV_20150310T134411...,None,None,data_imperial/S1A_OPER_AUX_POEORB_OPOD_2021030...,"MULTIPOLYGON (((-114.64096 32.61768, -114.6921..."
2015-04-03,2015-04-03 13:44:13,D,S1A,VV,1,data_imperial/S1A_IW_SLC__1SSV_20150403T134412...,data_imperial/S1A_IW_SLC__1SSV_20150403T134412...,None,None,data_imperial/S1A_OPER_AUX_POEORB_OPOD_2021030...,"MULTIPOLYGON (((-114.64328 32.60917, -114.6944..."
2015-04-27,2015-04-27 13:44:14,D,S1A,VV,1,data_imperial/S1A_IW_SLC__1SSV_20150427T134413...,data_imperial/S1A_IW_SLC__1SSV_20150427T134413...,None,None,data_imperial/S1A_OPER_AUX_POEORB_OPOD_2021030...,"MULTIPOLYGON (((-114.64313 32.60789, -114.6943..."
2015-05-21,2015-05-21 13:44:15,D,S1A,VV,1,data_imperial/S1A_IW_SLC__1SSV_20150521T134415...,data_imperial/S1A_IW_SLC__1SSV_20150521T134415...,None,None,data_imperial/S1A_OPER_AUX_POEORB_OPOD_2021030...,"MULTIPOLYGON (((-114.6423 32.60788, -114.69347..."


In [ ]:
sbas = Stack(WORKDIR, drop_if_exists=True).set_scenes(scenes).set_reference(REFERENCE)
sbas.to_dataframe()

NOTE: auto set reference scene 2015-01-21. You can change it like Stack.set_reference("2022-01-20")


,datetime,orbit,mission,polarization,subswath,datapath,metapath,noisepath,calibpath,orbitpath,geometry
date,,,,,,,,,,,
2015-01-21,2015-01-21 13:44:13,D,S1A,VV,1,data_imperial/S1A_IW_SLC__1SSV_20150121T134412...,data_imperial/S1A_IW_SLC__1SSV_20150121T134412...,None,None,data_imperial/S1A_OPER_AUX_POEORB_OPOD_2021030...,"MULTIPOLYGON (((-114.64126 32.6171, -114.69249..."
2015-03-10,2015-03-10 13:44:12,D,S1A,VV,1,data_imperial/S1A_IW_SLC__1SSV_20150310T134411...,data_imperial/S1A_IW_SLC__1SSV_20150310T134411...,None,None,data_imperial/S1A_OPER_AUX_POEORB_OPOD_2021030...,"MULTIPOLYGON (((-114.64096 32.61768, -114.6921..."
2015-04-03,2015-04-03 13:44:13,D,S1A,VV,1,data_imperial/S1A_IW_SLC__1SSV_20150403T134412...,data_imperial/S1A_IW_SLC__1SSV_20150403T134412...,None,None,data_imperial/S1A_OPER_AUX_POEORB_OPOD_2021030...,"MULTIPOLYGON (((-114.64328 32.60917, -114.6944..."
2015-04-27,2015-04-27 13:44:14,D,S1A,VV,1,data_imperial/S1A_IW_SLC__1SSV_20150427T134413...,data_imperial/S1A_IW_SLC__1SSV_20150427T134413...,None,None,data_imperial/S1A_OPER_AUX_POEORB_OPOD_2021030...,"MULTIPOLYGON (((-114.64313 32.60789, -114.6943..."
2015-05-21,2015-05-21 13:44:15,D,S1A,VV,1,data_imperial/S1A_IW_SLC__1SSV_20150521T134415...,data_imperial/S1A_IW_SLC__1SSV_20150521T134415...,None,None,data_imperial/S1A_OPER_AUX_POEORB_OPOD_2021030...,"MULTIPOLYGON (((-114.6423 32.60788, -114.69347..."


In [ ]:
sbas.plot_scenes()

### Load DEM

The function below loads DEM from file or Xarray variable and converts heights to ellipsoidal model using EGM96 grid.

In [ ]:
sbas.load_dem(DEM, AOI)

Save DEM on WGS84 Ellipsoid:   0%|          | 0/9000000.0 [00:00<?, ?it/s]

In [ ]:
sbas.plot_scenes()
plt.savefig('Estimated Scene Locations.jpg')

## Align a Stack of Images

In [ ]:
sbas.compute_align()

Preparing Reference:   0%|          | 0/1 [00:00<?, ?it/s]

Aligning Repeat:   0%|          | 0/4 [00:00<?, ?it/s]

## SBAS Baseline

In [ ]:
baseline_pairs = sbas.baseline_pairs(days=BASEDAYS, meters=BASEMETERS)
baseline_pairs

Note: function baseline_pairs() renamed to sbas_pairs(). Use separate filtering functions when needed.


,ref,rep,ref_baseline,rep_baseline,pair,baseline,duration,rel
0,2015-01-21,2015-03-10,52.62,-37.72,2015-01-21 2015-03-10,-90.34,48,NaT
1,2015-01-21,2015-04-03,52.62,-0.00,2015-01-21 2015-04-03,-52.62,72,NaT
2,2015-01-21,2015-04-27,52.62,-30.43,2015-01-21 2015-04-27,-83.05,96,NaT
3,2015-03-10,2015-04-03,-37.72,-0.00,2015-03-10 2015-04-03,37.72,24,NaT
4,2015-03-10,2015-04-27,-37.72,-30.43,2015-03-10 2015-04-27,7.29,48,NaT
5,2015-03-10,2015-05-21,-37.72,-50.96,2015-03-10 2015-05-21,-13.24,72,NaT
6,2015-04-03,2015-04-27,-0.00,-30.43,2015-04-03 2015-04-27,-30.43,24,NaT
7,2015-04-03,2015-05-21,-0.00,-50.96,2015-04-03 2015-05-21,-50.96,48,NaT
8,2015-04-27,2015-05-21,-30.43,-50.96,2015-04-27 2015-05-21,-20.53,24,NaT


In [ ]:
with mpl_settings({'figure.dpi': 150}):
    sbas.plot_baseline(baseline_pairs)
plt.savefig('Baseline.jpg')

## Geocoding

In [ ]:
# use default 60m coordinates grid
sbas.compute_geocode()

Radar Transform Computing:   0%|          | 0/1 [00:00<?, ?it/s]

Radar Transform Indexing:   0%|          | 0/9000000.0 [00:00<?, ?it/s]

INFO:distributed.core:Starting established connection to tcp://127.0.0.1:42002
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:38848
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:38864
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:38880


### DEM in Radar Coordinates

The grids are NetCDF files processing as xarray DataArrays.

In [ ]:
sbas.plot_topo()
plt.savefig('Topography on WGS84 ellipsoid, [m].jpg')

## Interferograms

Define a single interferogram or a SBAS series. Make direct and reverse interferograms (from past to future or from future to past).

Decimation is useful to save disk space. Geocoding results are always produced on the provided DEM grid so the output grid and resolution are the same to the DEM. By this way, ascending and descending orbit results are always defined on the same grid by design. An internal processing cell is about 30 x 30 meters size and for default output 60m resolution (like to GMTSAR and GAMMA software) decimation 2x2 is reasonable. For the default wavelength=200 for Gaussian filter 1/4 of wavelength is approximately equal to ~60 meters and better resolution is mostly useless (while it can be used for small objects detection). For wavelength=400 meters use 90m DEM resolution with decimation 4x4.

The grids are NetCDF files processing as xarray DataArrays.

In [ ]:
pairs = baseline_pairs[['ref', 'rep']]
pairs

,ref,rep
0,2015-01-21,2015-03-10
1,2015-01-21,2015-04-03
2,2015-01-21,2015-04-27
3,2015-03-10,2015-04-03
4,2015-03-10,2015-04-27
5,2015-03-10,2015-05-21
6,2015-04-03,2015-04-27
7,2015-04-03,2015-05-21
8,2015-04-27,2015-05-21


In [ ]:
# load Sentinel-1 data
data = sbas.open_data()

In [ ]:
# Gaussian filtering 400m cut-off wavelength with multilooking 1x4 on Sentinel-1 intensity
intensity = sbas.multilooking(np.square(np.abs(data)), wavelength=400, coarsen=(1,4))

In [ ]:
phase = sbas.multilooking(sbas.phasediff(pairs), wavelength=400, coarsen=(1,4))

In [ ]:
corr = sbas.correlation(phase, intensity)

In [ ]:
# Goldstein filter expects square grid cells produced using multilooking
intf_filt = sbas.interferogram(sbas.goldstein(phase, corr, 32))

In [ ]:
# use default 60m resolution
decimator = sbas.decimator()

In [ ]:
# compute together because correlation depends on phase, and filtered phase depends on correlation.
tqdm_dask(result := dask.persist(decimator(corr), decimator(intf_filt)), desc='Compute Phase and Correlation')
# unpack results
corr60m, intf60m = result

Compute Phase and Correlation:   0%|          | 0/9000000.0 [00:00<?, ?it/s]

In [ ]:
sbas.plot_interferograms(intf60m, cols=3, size=3, caption='Phase, [rad]')
plt.savefig('Phase, [rad].jpg')

In [ ]:
sbas.plot_correlations(corr60m, cols=3, size=3, caption='Correlation')
plt.savefig('Correlation.jpg')

## Unwrapping

Unwrapping process requires a lot of RAM and that's really RAM consuming when a lot of parallel proccesses running togeter. To limit the parallel processing tasks apply argument "n_jobs". The default value n_jobs=-1 means all the processor cores van be used. Also, use interferogram decimation above to produce smaller interferograms. And in addition a custom SNAPHU configuration can reduce RAM usage as explained below.

In [ ]:
CORRLIMIT = 0.075
tqdm_dask(unwrap := sbas.unwrap_snaphu(intf60m, corr60m.where(corr60m>=CORRLIMIT)).persist(),
          desc='SNAPHU Unwrapping')

SNAPHU Unwrapping:   0%|          | 0/9000000.0 [00:00<?, ?it/s]

In [ ]:
sbas.plot_phases(unwrap.phase, cols=3, size=3, caption='Unwrapped Phase, [rad]', quantile=[0.01, 0.99])
plt.savefig('Unwrapped Phase, [rad].jpg')

In [ ]:
tqdm_dask(unwrap_ll := sbas.ra2ll(unwrap.phase).persist(), desc='Geocoding')

Geocoding:   0%|          | 0/9000000.0 [00:00<?, ?it/s]

In [ ]:
sbas.plot_phases(unwrap_ll, cols=3, size=3, caption='Unwrapped Phase in Geographic Coordinates, [rad]', quantile=[0.01, 0.99])
plt.savefig('Unwrapped Phase in Geographic Coordinates, [rad].jpg')

### Detrend Unwrapped Phase

Remove trend and apply gaussian filter to fix ionospheric effects and solid Earh's tides.

In [ ]:
tqdm_dask(detrend := (unwrap.phase - sbas.gaussian(unwrap.phase, wavelength=60000)).persist(), desc='Detrending')

Detrending:   0%|          | 0/9000000.0 [00:00<?, ?it/s]

In [1]:
sbas.plot_phases(detrend, cols=3, size=3, caption='Detrended Unwrapped Phase, [rad]', quantile=[0.01, 0.99])
plt.savefig('Detrended Unwrapped Phase, [rad].jpg')

NameError: name 'sbas' is not defined

In [ ]:
detrend_subset = detrend.sel(x=slice(9000,12000), y=slice(1800,2700))
sbas.plot_phases(detrend_subset, cols=3, size=3, caption='Detrended Unwrapped Phase AOI, [rad]', quantile=[0.01, 0.99])
plt.savefig('Detrended Unwrapped Phase AOI, [rad].jpg')

### Calculate Displacement Using Coherence-Weighted Least-Squares Solution

In [ ]:
# calculate phase displacement in radians and convert to LOS displacement in millimeter
tqdm_dask(disp := sbas.los_displacement_mm(sbas.lstsq(detrend, corr60m)).persist(), desc='SBAS Computing')
# clean 1st zero-filled displacement map for better visualization
disp[0] = np.nan

Note: data chunk size (inf, inf) is too large for stack processing
Note: auto tune data chunk size to a half of NetCDF chunk: (256, 256)


SBAS Computing:   0%|          | 0/9000000.0 [00:00<?, ?it/s]

In [ ]:
sbas.plot_displacements(disp, cols=3, size=3, caption='Cumulative LOS Displacement, [mm]', quantile=[0.01, 0.99])
plt.savefig('Cumulative LOS Displacement, [mm].jpg')

In [ ]:
disp_subset = disp.sel(x=slice(9000,12000), y=slice(1800,2700))
sbas.plot_displacements(disp_subset, cols=3, size=3, caption='Cumulative LOS Displacement AOI, [mm]', quantile=[0.01, 0.99])
plt.savefig('Cumulative LOS Displacement AOI, [mm].jpg')

In [ ]:
# geocode subset on the full interferogram grid and crop a valid area only
tqdm_dask(disp_subset_ll := sbas.cropna(sbas.ra2ll(disp.sel(x=slice(9000,12000), y=slice(1800,2700)))).persist(),
          desc='SBAS Computing')

SBAS Computing:   0%|          | 0/9000000.0 [00:00<?, ?it/s]

In [ ]:
sbas.plot_displacements(disp_subset_ll, cols=3, size=3, caption='Cumulative LOS Displacement in Geographic Coordinates AOI, [mm]',
                        quantile=[0.01, 0.99])
plt.savefig('Cumulative LOS Displacement Geographic Coordinates AOI, [mm].jpg')

## Pixel Displacement

In [ ]:
# define point coordinates
lat = 32.43
lon = -115.15

# find nearest pixel to the defined coordinates
# first zero replaced by NaN so convert it back to zero
disp_ll_pixel = sbas.ra2ll(disp_subset).sel(lat=lat, lon=lon, method='nearest').fillna(0)

In [ ]:
disp_ll_pixel.plot.scatter('date')
disp_ll_pixel.plot(lw=0.25)
plt.title(f'Cumulative LOS Displacement, [mm]\nlat={lat}, lon={lon}', fontsize=18)
plt.xlabel('Date', fontsize=14)
plt.ylabel('Displacement, [mm]', fontsize=16)
plt.grid()
plt.savefig('Cumulative LOS Displacement POI, [mm].jpg')

## 3D Interactive Maps

In [ ]:
sbas.export_vtk(disp_subset_ll, 'disp_subset', mask='auto')

Exporting WGS84 VTK(s):   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# build interactive 3D plot
def load_mesh(plotter, index, offset=None):
    vtk_grid = pv.read(f'disp_subset.{index}.vtk')
    if offset:
        vtk_grid.points[:, 2] += offset
    vtk_grid = vtk_grid.scale([1, 1, 0.0002], inplace=True)
    plotter.add_mesh(vtk_grid, scalars='los', cmap='turbo', clim=(-30,10), opacity=0.5, ambient=0.5, label=str(index))

plotter = pv.Plotter(notebook=True)
load_mesh(plotter, 1, 0)
load_mesh(plotter, 2, 200)
load_mesh(plotter, 3, 400)
load_mesh(plotter, 4, 600)
plotter.show_axes()
plotter.show(screenshot='3D LOS Displacements Stack.png', jupyter_backend='panel', return_viewer=True)
plotter.add_title(f'Interactive LOS Displacements on DEM', font_size=32)
plotter._on_first_render_request()
panel.panel(
    plotter.render_window, orientation_widget=plotter.renderer.axes_enabled,
    enable_keybindings=False, sizing_mode='stretch_width', min_height=600
)

In [ ]:
plotter = pv.Plotter(shape=(2, 2), notebook=True)

def load_mesh(plotter, index):
    vtk_grid = pv.read(f'disp_subset.{index}.vtk').scale([1, 1, 0.0002], inplace=True)
    plotter.add_mesh(vtk_grid, scalars='los', cmap='turbo', clim=(-30,10), ambient=0.1, label=str(index))
    plotter.add_title(str(disp_subset_ll.isel(date=index).date.dt.date.item()), font_size=24)

plotter.subplot(0, 0)
load_mesh(plotter, 1)

plotter.subplot(0, 1)
load_mesh(plotter, 2)

plotter.subplot(1, 0)
load_mesh(plotter, 3)
plotter.show_axes()

plotter.subplot(1, 1)
load_mesh(plotter, 4)

plotter.subplot(0, 1)
plotter.add_legend(loc='upper center')
plotter.show_axes()
plotter.show(screenshot='3D LOS Displacements Grid.png', jupyter_backend='panel', return_viewer=True)
plotter._on_first_render_request()
panel.panel(
    plotter.render_window, orientation_widget=plotter.renderer.axes_enabled,
    enable_keybindings=False, sizing_mode='stretch_width', min_height=600
)

## 2D Interactive Map

The map is also available as a standalone web page, which can be saved and used locally or shared on any web platform: https://insar.dev/ui/Imperial_Valley_2015.html

In [ ]:
# prepare and decimate data
velocity_stacked = sbas.velocity(disp_subset.stack(stack=['y','x'])).rename('velocity')
df = sbas.ra2ll(velocity_stacked).to_dataframe().dropna()
gdf = gpd.GeoDataFrame(df[['lat','lon','velocity']], geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326)
gdf

lat         lon   velocity                     geometry
y      x                                                                     
1800.0 9002.0   32.450201 -115.105556 -32.694195   POINT (-115.10556 32.4502)
       9018.0   32.450201 -115.106285 -31.706636   POINT (-115.10629 32.4502)
       9034.0   32.450201 -115.107014 -27.238071   POINT (-115.10701 32.4502)
       9050.0   32.450201 -115.107743 -23.427515   POINT (-115.10774 32.4502)
       9066.0   32.450219 -115.108472 -17.282242  POINT (-115.10847 32.45022)
...                   ...         ...        ...                          ...
2700.0 11930.0  32.358995 -115.258192  -7.344057  POINT (-115.25819 32.35899)
       11946.0  32.359031 -115.258895  -8.511416   POINT (-115.2589 32.35903)
       11962.0  32.359379 -115.259377  -6.892149  POINT (-115.25938 32.35938)
       11978.0  32.359379 -115.260106  -1.104134  POINT (-115.26011 32.35938)
       11994.0  32.359379 -115.260835   4.113772  POINT (-115.26083 32.35938)

[42488 rows x 4 columns]

In [ ]:
# specify pixel boundaries in geo coordinates
def point_to_rectangle(row):
    #print (row)
    import shapely
    return shapely.geometry.Polygon([
        (row._lon0, row._lat0),
        (row._lon1, row._lat1),
        (row._lon2, row._lat2),
        (row._lon3, row._lat3)
    ])

dycell = np.array([-0.5, -0.5, 0.5, 0.5])
dxcell = np.array([-0.5, 0.5, 0.5, -0.5])
# use the map pixel spacing
cellsize = (4, 16)
for idx, dydx in enumerate(zip(dycell*cellsize[0], dxcell*cellsize[1])):
    index = pd.MultiIndex.from_tuples(
        [(y + dydx[0], x + dydx[1]) for y, x in gdf.index],
        names=gdf.index.names
    )
    coords = xr.Dataset(coords={'stack': index})
    coords_dydx = sbas.ra2ll(coords).to_dataframe()[['lat','lon']]
    gdf[f'_lat{idx}'] = coords_dydx.lat.values
    gdf[f'_lon{idx}'] = coords_dydx.lon.values
gdf['geometry'] = gdf.apply(lambda row: point_to_rectangle(row), axis=1)
for col in gdf.columns[gdf.columns.str.contains('_')]:
    del gdf[col]
gdf

lat         lon   velocity  \
y      x                                           
1800.0 9002.0   32.450201 -115.105556 -32.694195   
       9018.0   32.450201 -115.106285 -31.706636   
       9034.0   32.450201 -115.107014 -27.238071   
       9050.0   32.450201 -115.107743 -23.427515   
       9066.0   32.450219 -115.108472 -17.282242   
...                   ...         ...        ...   
2700.0 11930.0  32.358995 -115.258192  -7.344057   
       11946.0  32.359031 -115.258895  -8.511416   
       11962.0  32.359379 -115.259377  -6.892149   
       11978.0  32.359379 -115.260106  -1.104134   
       11994.0  32.359379 -115.260835   4.113772   

                                                         geometry  
y      x                                                           
1800.0 9002.0   POLYGON ((-115.10519 32.45021, -115.10592 32.4...  
       9018.0   POLYGON ((-115.10592 32.45039, -115.10665 32.4...  
       9034.0   POLYGON ((-115.10665 32.45043, -115.10738 32.4...  
       9050.0   POLYGON ((-115.10738 32.45046, -115.10811 32.4...  
       9066.0   POLYGON ((-115.10811 32.45046, -115.10884 32.4...  
...                                                           ...  
2700.0 11930.0  POLYGON ((-115.25781 32.35921, -115.25828 32.3...  
       11946.0  POLYGON ((-115.25828 32.35957, -115.25901 32.3...  
       11962.0  POLYGON ((-115.25901 32.35957, -115.25974 32.3...  
       11978.0  POLYGON ((-115.25974 32.35961, -115.26047 32.3...  
       11994.0  POLYGON ((-115.26047 32.35964, -115.2612 32.35...  

[42488 rows x 4 columns]

In [ ]:
# group the data by decimated lat/lon
gdf['lat_group'] = (gdf['lat'] // 0.002).astype(int)
gdf['lon_group'] = (gdf['lon'] // 0.002).astype(int)
# merge geometries and average velocities and lat/lon
gdf = gdf.groupby(['lat_group', 'lon_group']).agg({
    'lat': 'mean',
    'lon': 'mean',
    'velocity': 'mean',
    'geometry': lambda x: shapely.ops.unary_union(x)
}).reset_index()
# drop the lat_group and lon_group columns used for grouping
gdf = gdf.drop(columns=['lat_group', 'lon_group'])
gdf

,lat,lon,velocity,geometry
0,32.339890,-115.140913,-31.883810,"POLYGON ((-115.14164 32.33989, -115.14164 32.3..."
1,32.339813,-115.138908,-34.390587,"POLYGON ((-115.138 32.33956, -115.13873 32.339..."
2,32.339659,-115.136904,-21.449905,"POLYGON ((-115.13581 32.33911, -115.13654 32.3..."
3,32.339611,-115.135081,-24.113037,"POLYGON ((-115.13508 32.33911, -115.13581 32.3..."
4,32.339418,-115.133259,-14.407969,"POLYGON ((-115.13289 32.33886, -115.13362 32.3..."
...,...,...,...,...
3901,32.468576,-115.220739,-16.968019,"POLYGON ((-115.21965 32.46842, -115.21991 32.4..."
3902,32.468004,-115.219529,-18.599262,"POLYGON ((-115.21892 32.46842, -115.21965 32.4..."
3903,32.470415,-115.236172,-21.359385,"POLYGON ((-115.23632 32.46951, -115.23667 32.4..."
3904,32.470374,-115.234895,-2.631674,"POLYGON ((-115.2349 32.46946, -115.23559 32.46..."


In [ ]:
# easy way to create GeoJSON, but it may fail for complex geometries
# with open('Imperial_Valley_2015.geojson', 'w') as f:
#     f.write(gdf.head(1).to_json())

# convert the GeoDataFrame to a GeoJSON-like Python dictionary
geojson_dict = {
    "type": "FeatureCollection",
    "features": []
}
 # convert Shapely geometry to GeoJSON format
for _, row in gdf.iterrows():
    feature = {
        "type": "Feature",
        "geometry": shapely.geometry.mapping(row['geometry']),
        "properties": {
            "lat": row['lat'],
            "lon": row['lon'],
            "velocity": row['velocity']
        }
    }
    geojson_dict["features"].append(feature)

# write the dictionary to a GeoJSON file
with open('Imperial_Valley_2015.geojson', 'w') as f:
    json.dump(geojson_dict, f, indent=2)

In [ ]:
# load the GeoJSON from the file
with open('Imperial_Valley_2015.geojson', 'r') as f:
    geojson = json.load(f)
print ('Pixels loaded:', len(geojson['features']))

Pixels loaded: 3906


In [ ]:
# Create a colormap for velocities
colormap = plt.get_cmap('turbo')
def velocity_to_color(velocity, limits=[-60, 60]):
    """Convert velocity to a color from the colormap."""
    normalized = (velocity - limits[0]) / (limits[1] - limits[0])
    return matplotlib.colors.to_hex(colormap(normalized))

# Embed styles and popup content directly into GeoJSON properties
for feature in geojson['features']:
    color = velocity_to_color(feature['properties']['velocity'])
    feature['properties']['style'] = {
        'color': color,
        'weight': 1,
        'fillColor': color,
        'fillOpacity': 0.5
    }

# Initialize the map
location = [32.40318, -115.18128]

esri_layer = TileLayer(
    url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attribution='Esri',
    name='Esri Satellite',
    max_zoom=20,
    max_native_zoom=19,
    base=True  # Set Esri as the base layer
)
# Add OpenStreetMap layer with a max zoom of 20, but not as base
osm_layer = TileLayer(
    url='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
    attribution='OpenStreetMap',
    name='OpenStreetMap',
    max_zoom=20,
    max_native_zoom=19,
    base=True
)
m = Map(center=location, zoom=13, layers=[osm_layer], scroll_wheel_zoom=True, layout=dict(height='820px'))
m.add_layer(esri_layer)

# Define the GeoJSON layer with embedded styles
geo_json = GeoJSON(
    data=geojson,
    style_callback=lambda feature: feature['properties']['style'],
    hover_style={'fillOpacity': 1},
    point_style={'weight': 0.5, 'fillOpacity': 0.5},
    name='InSAR'
)

# Add GeoJSON layer to the map
m.add_layer(geo_json)

# Add layer control to switch between layers
m.add_control(LayersControl(position='topright'))

# Display the map
display(m)

In [ ]:
# save the interactive map as an HMTL file
m.save('Imperial_Valley_2015.html', title='InSAR Velocity Map')